In [ ]:
!pip install transformers gradio torch


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
import gradio as gr
import torch

# --- Load Simplify Model ---
simplify_model_path = "/content/drive/MyDrive/legal_simplifier/final_model"
simplify_tokenizer = AutoTokenizer.from_pretrained(simplify_model_path)
simplify_model = AutoModelForSeq2SeqLM.from_pretrained(simplify_model_path)

# --- Load Punjabi Translation Model ---
punjabi_model_path = "/content/drive/MyDrive/punjabi_model"
punjabi_tokenizer = AutoTokenizer.from_pretrained(punjabi_model_path)
punjabi_model = AutoModelForSeq2SeqLM.from_pretrained(punjabi_model_path)

malayalam_model_path = "/content/drive/MyDrive/en-ml-finetuned-60k"
malayalam_tokenizer = AutoTokenizer.from_pretrained(malayalam_model_path)
malayalam_model = AutoModelForSeq2SeqLM.from_pretrained(malayalam_model_path)

# Available languages
languages = ["Punjabi", "Gujarati", "Malayalam", "Tamil"]

# --- Simplify & Translate Function ---
def simplify_and_translate(text, lang):
    if not text.strip():
        return "Please enter some text.", ""

    # 1 Simplify
    inputs = simplify_tokenizer(text, return_tensors="pt", truncation=True)
    simplified_ids = simplify_model.generate(**inputs, max_length=512)
    simplified_text = simplify_tokenizer.decode(simplified_ids[0], skip_special_tokens=True)

    # 2 Translate
    if lang == "Punjabi":
        inputs = punjabi_tokenizer(text, return_tensors="pt", truncation=True)
        with torch.no_grad():
            outputs = punjabi_model.generate(**inputs, max_length=512)
        translated_text = punjabi_tokenizer.decode(outputs[0], skip_special_tokens=True)
    elif lang == "Malayalam":
        inputs = malayalam_tokenizer(text, return_tensors="pt", truncation=True)
        with torch.no_grad():
            outputs = malayalam_model.generate(**inputs, max_length=512)
        translated_text = malayalam_tokenizer.decode(outputs[0], skip_special_tokens=True)
    else:
        translated_text = f"Translation for {lang} not available yet."

    return translated_text

# --- Gradio Interface ---
demo = gr.Interface(
    fn=simplify_and_translate,
    inputs=[gr.Textbox(label="Enter English Text"), gr.Dropdown(languages, label="Choose Language")],
    outputs=[gr.Textbox(label="Translated Text")],
    title="Legal Text Simplifier & Translator",
    description="Simplifies your text first, then translates it to selected language."
)

demo.launch(share=True)


/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6c2bf828172739813b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
